# LoRA layer-wise factual predictions — Colab launcher

Thin launcher: all logic lives in the repo; this notebook only clones and runs it.

Setup: **Runtime → Change runtime type → A100 GPU** (fp32 measurement passes are
affordable on A100; on a T4 they are ~8x slower — use an A100).

Outputs go to Google Drive so a disconnected session can resume: every stage
reads/writes artifacts under `output_dir`, so re-running with
`STAGES = "analyze,patch,..."` continues where the last session stopped.

In [ ]:
REF = "logic_fixes"               # branch for iteration, or a commit SHA to pin a run
CONFIG = "configs/default.yaml"   # configs/dev.yaml for a fast smoke test
STAGES = "all"                    # or e.g. "analyze,patch" to resume a session

from google.colab import drive
drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/nlp_outputs"   # persists across sessions

import os
if os.path.exists("nlp-project"):
    !cd nlp-project && git fetch && git checkout {REF} && git pull --ff-only || true
else:
    !git clone --branch {REF} https://github.com/hadasy-tau/nlp-project.git

In [ ]:
!pip install -q -r nlp-project/requirements.txt
!pip install -q -e nlp-project

import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — enable a GPU!"
print("GPU:", name)
if "A100" not in name:
    print("WARNING: not an A100 — the fp32 measurement passes will be slow on this GPU.")

In [ ]:
# Model and data are configurable — add --set overrides as needed, e.g.:
#   --set model.name=EleutherAI/pythia-1b-deduped
!cd nlp-project && python -m lora_lens.run --config {CONFIG} --stages {STAGES} \
    --set output_dir={OUTPUT_DIR}

In [ ]:
# Bundle the small artifacts for download (results, figures, config, training log);
# adapters stay on Drive.
!cd {OUTPUT_DIR} && zip -qr /content/results.zip results figures config_resolved.yaml lora/training_log.csv
print("results.zip is in the Colab file browser (left sidebar) — or just use Drive:", OUTPUT_DIR)